# Vibration Isolation — FC/IMU and Payload
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

The single EDF is a high-RPM rotating machine on a stiff CFRP monocoque. Its
dominant forcing is the **1-per-rev shaft imbalance** at the shaft frequency
(`f_shaft = RPM / 60`, derived from the same `rpm_from_diameter` law the rest
of the pipeline uses); blade-pass (`n_blades × f_shaft`) is far higher. That
1/rev line corrupts the IMU attitude estimate and blurs the EO payload.

This notebook soft-mounts the **FC/IMU cluster** and the **payload** on
vibration isolators and sizes each as a 1-DOF base-excitation isolator:
corner frequency, transmissibility at the forcing frequency, static sag,
sway/rattle space, isolator stiffness, and hardware mass. The sway space is
handed to `fuselage_design` (it costs packing volume); the isolator hardware
is carved into the mass budget.

Because the forcing is far above any practical isolator corner frequency, a
soft mount at f_n ≈ 40–60 Hz achieves large attenuation with only a fraction
of a millimetre of static deflection — so the packing-volume cost is small.

## Inputs

- `config/vibration.yaml` — isolation target, damping, shock load, isolated
  FC/IMU mass, isolator counts/mass.
- rotor RPM (`rpm_from_diameter`) + `config/prop_geometry.yaml` `n_blades`
  → forcing frequency.
- mission payload mass (soft-mounted whole).

## Outputs

- `out/vibration.yaml`
- `out/vibration_transmissibility.png`

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import yaml

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import solve_design_point
from conceptual_design.vtol_power import VTOLParams, rpm_from_diameter
from conceptual_design.vibration_isolation import (
    VibrationParams, size_isolation, write_vibration_yaml,
)
from conceptual_design import reports
from conceptual_design.plots import plot_transmissibility

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` (same pattern as NB2–NB4) for the
payload mass and the rotor, and derive the EDF forcing frequency.

---

In [ ]:
# -- Re-run the sizing loop; rotor 1/rev forcing from the EDF shaft speed ------
inp, result = solve_design_point(CONFIG_PATH)
vib_p = VibrationParams.from_yaml(CONFIG_PATH / "vibration.yaml")

with open(CONFIG_PATH / "prop_geometry.yaml") as f:
    n_blades = int(yaml.safe_load(f)["n_blades"])

vtol = VTOLParams.from_propulsive(inp.prop, inp.env)
rpm  = rpm_from_diameter(inp.rotor.D_rotor_m, vtol)

print(f"EDF shaft speed   : {rpm:.0f} RPM  -> {rpm/60:.1f} Hz (1/rev)")
print(f"Blade count       : {n_blades}  -> blade-pass {rpm/60*n_blades:.0f} Hz")
print(f"Payload (isolated): {result.m_payload_kg:.3f} kg")
print(f"FC/IMU (isolated) : {vib_p.m_fc_imu_kg:.3f} kg  "
      f"(subset of avionics {result.m_avionics_kg:.3f} kg)")
print(f"Target            : T <= {vib_p.target_transmissibility:.2f} "
      f"({100*(1-vib_p.target_transmissibility):.0f}% attenuation), "
      f"zeta = {vib_p.damping_ratio:.2f}")

# Section 2 — Isolator Sizing

Size the corner frequency `f_n` as high (stiffest, least sway) as still meets
the target transmissibility at the 1/rev forcing, check it sits in the valid
window (above the control bandwidth, below `f_forcing/√2`), and size the
sway space and isolator stiffness/mass for each module.

---

In [ ]:
res = size_isolation(rpm, n_blades, result.m_payload_kg, vib_p)

lo, hi = res["f_n_window_hz"]
print(f"Forcing (1/rev)   : {res['f_shaft_hz']:.1f} Hz")
print(f"Corner freq f_n   : {res['f_n_hz']:.1f} Hz   "
      f"(valid window {lo:.0f}-{hi:.0f} Hz -> {'OK' if res['window_ok'] else 'OUT OF WINDOW'})")
print(f"Sway per bay      : {res['sway_mm']:.2f} mm  "
      f"(shock {vib_p.shock_load_g:.0f} g)")
print()
reports.print_isolation_table(res)

assert res["window_ok"], "corner frequency outside the valid isolation window"

# Section 3 — Transmissibility Curve

The transmissibility of the sized isolator vs. frequency ratio, with the
1/rev and blade-pass forcing lines and the target. Both forcing lines land
in the isolation region (`r > √2`); the design is set at the 1/rev line
(the harder, lower-frequency case).

---

In [ ]:
plot_transmissibility(res, vib_p, OUT_PATH / "vibration_transmissibility.png")

# Section 4 — Output Export

`out/vibration.yaml` — consumed by `fuselage_design` (sway pad fed into the
bay stack; isolator hardware carved from the avionics/structural fractions)
and read for reference downstream.

---

In [ ]:
write_vibration_yaml(res, vib_p, OUT_PATH / "vibration.yaml")
print(f"Vibration design written -> {OUT_PATH / 'vibration.yaml'}")

# Section 5 — Design Summary

---

In [ ]:
reports.print_vibration_summary(res, vib_p)